In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../')
from datasets import load_dataset, load_from_disk
from hydra import compose, initialize
from utils.configuration import list_detector_model_names, list_hydra_group_options
with initialize(version_base=None, config_path="../conf"):
    cfg = compose(
        config_name="config",
        overrides=["detector=bert"]
    )
    
detectors = list_detector_model_names("../conf", "detector")
print(detectors)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


/home/gx1/git/when-human-write-like-llm/when-human-write-like-machines/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['BERT-Defense', 'binoculars-falcon-7b', 'FastDetectGPT', 'radar', 'RoBERTa-Defense']


## Baseline detection performance


In [2]:
from pathlib import Path

from utils.evaluation.Metrics import Metrics
from utils.evaluation.ResultsReportBuilder import build_detector_table

results_dir = Path(".." / Path(cfg.experiment.paths.results_dir) )
build_detector_table(results_dir, 
                                      detectors, 
                                      baseline_regime="free_llm",
                                      n_bootstrap=100, 
                                      random_seed=42)

/home/gx1/git/when-human-write-like-llm/when-human-write-like-machines/venv/lib/python3.12/site-packages/pandera/_pandas_deprecated.py:144: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


,AUROC FREE-LLM,TPR@1%FPR
,mean ± 95% CI,mean ± 95% CI
Detector,,
BERT-Defense,"0.246 [0.232, 0.262]","0.002 [0.000, 0.005]"
binoculars-falcon-7b,"0.996 [0.995, 0.998]","0.973 [0.964, 0.981]"
FastDetectGPT,"0.998 [0.997, 0.999]","0.974 [0.960, 0.987]"
radar,"0.990 [0.986, 0.992]","0.802 [0.679, 0.873]"
RoBERTa-Defense,"0.640 [0.619, 0.659]","0.026 [0.002, 0.076]"


## 2 Robustness under H2L rewriting

In [3]:

from utils.evaluation.ResultsReportBuilder import build_h2l_robustness_table


detector_table = build_h2l_robustness_table(results_dir, detectors, n_bootstrap=100, random_seed=42)

In [4]:
detector_table

,AUROC H2L,ΔAUROC H2L,TPR@1%FPR H2L,ΔTPR@1%FPR H2L
,mean [95% CI],mean [95% CI],mean [95% CI],mean [95% CI]
Detector,,,,
BERT-Defense,"0.472 [0.452, 0.491]","0.225 [0.203, 0.243]","0.011 [0.004, 0.019]","0.009 [0.002, 0.017]"
binoculars-falcon-7b,"0.727 [0.713, 0.742]","-0.269 [-0.283, -0.254]","0.127 [0.086, 0.162]","-0.846 [-0.881, -0.810]"
FastDetectGPT,"0.901 [0.891, 0.909]","-0.097 [-0.107, -0.089]","0.382 [0.323, 0.435]","-0.591 [-0.650, -0.540]"
radar,"0.639 [0.617, 0.660]","-0.351 [-0.372, -0.332]","0.099 [0.065, 0.125]","-0.703 [-0.757, -0.606]"
RoBERTa-Defense,"0.677 [0.654, 0.697]","0.037 [0.017, 0.058]","0.026 [0.004, 0.063]","-0.000 [-0.023, 0.019]"


## 4.3 Robustness under LLM2L rewriting

In [6]:


from utils.evaluation.ResultsReportBuilder import build_llm2l_robustness_table


build_llm2l_robustness_table(results_dir, detectors, n_bootstrap=100, random_seed=42)

# from utils.evaluation.BootstrapMetrics import bootstrap_metrics


# bootstrap_metrics = bootstrap_metrics(metrics, cfg.experiment.evaluation.bootstrap.n)

,AUROC LLM2L,ΔAUROC LLM2L,TPR@1%FPR LLM2L,ΔTPR@1%FPR LLM2L
,mean [95% CI],mean [95% CI],mean [95% CI],mean [95% CI]
Detector,,,,
BERT-Defense,"0.283 [0.269, 0.298]","0.036 [0.031, 0.043]","0.003 [0.001, 0.007]","0.002 [-0.001, 0.004]"
binoculars-falcon-7b,"0.984 [0.980, 0.987]","-0.012 [-0.016, -0.009]","0.903 [0.867, 0.920]","-0.070 [-0.100, -0.057]"
FastDetectGPT,"0.989 [0.987, 0.992]","-0.009 [-0.011, -0.006]","0.875 [0.835, 0.916]","-0.094 [-0.118, -0.067]"
radar,"0.988 [0.984, 0.990]","-0.002 [-0.003, -0.001]","0.804 [0.691, 0.873]","0.002 [-0.018, 0.023]"
RoBERTa-Defense,"0.642 [0.621, 0.664]","0.002 [-0.007, 0.012]","0.020 [0.000, 0.060]","-0.006 [-0.025, 0.002]"
